## Download the dataset (title + abstract)

The titles are in the paper folder while the abstracts are in the abstract folder.


In [ ]:
"""
This code downloads and processes academic paper data from the Semantic Scholar API. Here's what it does:

1. Authentication:
- Uses an API key to authenticate with Semantic Scholar API
- Creates a local output directory './s2ag_dataset_title_abstracts'

2. Data Download:
- Gets the latest data release ID
- Downloads two datasets: 'papers' and 'abstracts' 
- Saves the downloaded files as gzipped files in dataset-specific subdirectories:
  - ./s2ag_dataset_title_abstracts/papers/papers_*.gz
  - ./s2ag_dataset_title_abstracts/abstracts/abstracts_*.gz

3. Data Processing:
- Extracts titles and abstracts specifically for Computer Science papers
- Matches papers with their abstracts using corpusid
- Creates a pandas DataFrame with matched title-abstract pairs

4. Output Files:
- Gzipped raw data files in the papers/ and abstracts/ subdirectories
- Final CSV file (csv_output_path) containing:
  - Column 'title': Paper titles
  - Column 'abstract': Corresponding paper abstracts
  - Only includes Computer Science papers that have both title and abstract

Note: The csv_output_path variable appears to be undefined in the code snippet,
but the final CSV would contain the processed title-abstract pairs.
"""

import requests
import json
import os
import gzip
import pandas as pd

# Set your S2 API key
api_key = ""

# Headers for authentication
headers = {
    "x-api-key": api_key
}

# Base output directory locally
base_output_dir = "./s2ag_dataset_title_abstracts"
os.makedirs(base_output_dir, exist_ok=True)


# Step 1: Get the latest release ID
latest_release_url = "https://api.semanticscholar.org/datasets/v1/release/latest"
response = requests.get(latest_release_url, headers=headers)
if response.status_code != 200:
    print(f"Failed to fetch latest release: {response.status_code} - {response.text}")
    exit()

latest_release = response.json()
release_id = latest_release["release_id"]
print("Latest release ID:", release_id)

# Step 2: Datasets to download
datasets = ["papers", "abstracts"]

# Dictionary to store data
data = {"title": [], "abstract": []}

for dataset_name in datasets:
    # Create dataset-specific directory
    output_dir = os.path.join(base_output_dir, dataset_name)
    os.makedirs(output_dir, exist_ok=True)

    # Step 3: Get metadata for the dataset
    dataset_url = f"https://api.semanticscholar.org/datasets/v1/release/{release_id}/dataset/{dataset_name}"
    response = requests.get(dataset_url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to fetch metadata for {dataset_name}: {response.status_code} - {response.text}")
        continue

    dataset_info = response.json()
    print(f"\n{dataset_name.capitalize()} dataset metadata:")
    print(json.dumps(dataset_info, indent=2))

    # Step 4: Download all dataset files with simplified file names
    for index, file_url in enumerate(dataset_info["files"]):  # Process all files
        # Generate a shorter file name (e.g., papers_0.gz, abstracts_1.gz)
        file_name = f"{dataset_name}_{index}.gz"
        output_path = os.path.join(output_dir, file_name)

        print(f"Downloading {dataset_name}/{file_name}...")
        file_response = requests.get(file_url, headers=headers, stream=True)

        if file_response.status_code == 200:
            with open(output_path, "wb") as f:
                for chunk in file_response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"Saved {file_name} to {output_path}")
        else:
            print(f"Failed to download {file_name}: {file_response.status_code} - {file_response.text}")

# Step 5: Process the datasets to extract titles and abstracts for Computer Science papers
print("\nProcessing datasets to extract titles and abstracts for Computer Science papers...")

# Dictionary to store titles and abstracts by corpusid
title_dict = {}
abstract_dict = {}

# Process papers dataset (for titles and field of study)
papers_dir = os.path.join(base_output_dir, "papers")
if os.path.exists(papers_dir):
    for file_name in os.listdir(papers_dir):
        if file_name.endswith(".gz"):
            file_path = os.path.join(papers_dir, file_name)
            try:
                with gzip.open(file_path, "rt", encoding="utf-8") as f:
                    for line in f:
                        try:
                            record = json.loads(line.strip())
                            corpusid = record.get("corpusid")
                            title = record.get("title")
                            fields_of_study = record.get("fieldsOfStudy", [])
                            # Filter for Computer Science
                            if corpusid and title and "Computer Science" in fields_of_study:
                                title_dict[corpusid] = title
                        except json.JSONDecodeError as e:
                            print(f"Error decoding JSON in {file_name}: {e}")
            except Exception as e:
                print(f"Error reading {file_name}: {e}")
else:
    print("Papers directory not found. Skipping paper processing.")

# Process abstracts dataset
abstracts_dir = os.path.join(base_output_dir, "abstracts")
if os.path.exists(abstracts_dir):
    for file_name in os.listdir(abstracts_dir):
        if file_name.endswith(".gz"):
            file_path = os.path.join(abstracts_dir, file_name)
            try:
                with gzip.open(file_path, "rt", encoding="utf-8") as f:
                    for line in f:
                        try:
                            record = json.loads(line.strip())
                            corpusid = record.get("corpusid")
                            abstract = record.get("abstract")
                            if corpusid and abstract:
                                abstract_dict[corpusid] = abstract
                        except json.JSONDecodeError as e:
                            print(f"Error decoding JSON in {file_name}: {e}")
            except Exception as e:
                print(f"Error reading {file_name}: {e}")
else:
    print("Abstracts directory not found. Skipping abstract processing.")

# Step 6: Combine titles and abstracts by corpusid for Computer Science papers
for corpusid in title_dict:
    if corpusid in abstract_dict:
        data["title"].append(title_dict[corpusid])
        data["abstract"].append(abstract_dict[corpusid])

# Step 7: Create DataFrame and save to CSV
if data["title"]:
    df = pd.DataFrame(data)
    print("\nSample of DataFrame (Computer Science papers):")
    print(df.head())

    # Save DataFrame to CSV
    df.to_csv(csv_output_path, index=False, encoding="utf-8")
    print(f"\nSaved DataFrame to {csv_output_path}")
else:
    print("\nNo matching title and abstract pairs found for Computer Science papers. CSV not created.")

print("\nProcessing complete!")

## DATASET CONSTRUCTION
Combine the datasets and filter English Only papers


In [ ]:
import gzip
import json
import os
import re

# This code processes academic paper titles and abstracts from gzipped JSON files, filtering for English content
# It has a 3-step pipeline: process titles, process abstracts, and merge them into a single dataset


# --- Config ---
papers_dir = "./s2ag_dataset_title_abstracts/papers"
abstracts_dir = "./s2ag_dataset_title_abstracts/abstracts"
output_dir = r"./paper_dataset"

titles_output_file = os.path.join(output_dir, "clean_titles.txt")
abstracts_output_file = os.path.join(output_dir, "clean_abstracts.txt")
merged_output_file = os.path.join(output_dir, "paper_dataset.txt")

buffer_size = 5000  # Optimal for performance

# --- Regex Patterns ---
LATEX_PATTERN = re.compile(r'\\usepackage|\\documentclass|\\begin\{document\}|\\cite\{|\\ref\{')
NON_ENGLISH_UNICODE_PATTERN = re.compile(r'[٠-٩\u0600-\u06FF\u4e00-\u9fff\uac00-\ud7af]')

def is_strictly_english(text: str, min_length: int = 10, min_words: int = 2) -> bool:
    """
    Check if a given text is strictly English and meets minimum quality criteria.

    Args:
        text (str): The input text to check
        min_length (int, optional): Minimum required length of text. Defaults to 10.
        min_words (int, optional): Minimum number of common English words required. Defaults to 2.

    Returns:
        bool: True if text is valid English, False otherwise

    The function performs the following checks:
    1. Verifies text meets minimum length requirement
    2. Checks for absence of non-English Unicode characters
    3. Ensures text doesn't contain LaTeX markup
    4. Confirms presence of minimum number of common English words
    """

    if not text or len(text) < min_length:
        return False
    if NON_ENGLISH_UNICODE_PATTERN.search(text):
        return False
    if LATEX_PATTERN.search(text.lower()):
        return False
    text_sample = f" {text.lower()[:200]} "
    common_words = {
        "the", "this", "that", "we", "our", "an", "a", "and", "is", "are",
        "for", "with", "from", "by", "on", "of", "in", "to", "using", "can",
        "have", "has", "as", "be", "based", "new", "approach", "method",
        "study", "paper", "research", "results", "present", "analysis",
        "model", "data", "system", "algorithm", "network", "learning",
        "detection", "classification", "prediction", "optimization",
        "propose", "show", "demonstrate", "evaluate", "performance",
        "experimental", "implementation", "framework", "problem", "solution"
    }
    return sum(1 for word in common_words if f" {word} " in text_sample) >= min_words

# --- Step 1: Process Titles ---
def process_titles():
    """
    Process and filter English titles from gzipped JSON files.
    
    This function:
    1. Creates the output directory if it doesn't exist
    2. Reads gzipped JSON files containing paper titles
    3. Filters titles to keep only valid English content
    4. Writes filtered titles to an output file in batches
    
    The function uses buffered writing for performance optimization and includes
    error handling for file operations and JSON parsing.
    
    Global variables used:
    - papers_dir: Directory containing input gzipped JSON files
    - output_dir: Directory for output files
    - titles_output_file: Path for the output file containing filtered titles
    - buffer_size: Size of write buffer for performance optimization
    
    Returns:
        None. Results are written directly to titles_output_file.
        Progress and completion messages are printed to stdout.
    
    Raises:
        No exceptions are raised - errors are caught and logged to stdout.
    """

    print(" Processing English titles...")
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(papers_dir) if f.endswith(".gz")]
    total_written = 0
    buffer = []

    with open(titles_output_file, "w", encoding="utf-8", buffering=16384) as out_f:
        for file in files:
            file_path = os.path.join(papers_dir, file)
            try:
                with gzip.open(file_path, "rt", encoding="utf-8") as f:
                    for line in f:
                        try:
                            record = json.loads(line.strip())
                            title = record.get("title")
                            if title and isinstance(title, str):
                                title = title.strip()
                                if title and is_strictly_english(title):
                                    buffer.append(title + "\n")
                                    total_written += 1
                        except json.JSONDecodeError:
                            continue

                        if len(buffer) >= buffer_size:
                            out_f.writelines(buffer)
                            buffer.clear()
            except Exception as e:
                print(f" Error reading {file}: {e}")
                continue

        if buffer:
            out_f.writelines(buffer)

    print(f" Step 1 complete: {total_written:,} titles written to {titles_output_file}")

# --- Step 2: Process Abstracts ---
def process_abstracts():
    """
    Process and filter English abstracts from gzipped JSON files.
    
    This function:
    1. Creates the output directory if it doesn't exist
    2. Reads gzipped JSON files containing paper abstracts
    3. Filters abstracts to keep only valid English content
    4. Writes filtered abstracts to an output file in batches
    
    The function uses buffered writing for performance optimization and includes
    error handling for file operations and JSON parsing.
    
    Global variables used:
    - abstracts_dir: Directory containing input gzipped JSON files
    - output_dir: Directory for output files
    - abstracts_output_file: Path for the output file containing filtered abstracts
    - buffer_size: Size of write buffer for performance optimization
    
    Returns:
        None. Results are written directly to abstracts_output_file.
        Progress and completion messages are printed to stdout.
    
    Raises:
        No exceptions are raised - errors are caught and logged to stdout.
        
    Note:
        Abstracts must meet stricter criteria than titles:
        - Minimum length of 20 characters
        - Contains at least 3 common English words
    """

    print("\n Processing English abstracts...")
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(abstracts_dir) if f.endswith(".gz")]
    total_written = 0
    buffer = []

    with open(abstracts_output_file, "w", encoding="utf-8", buffering=16384) as out_f:
        for file in files:
            file_path = os.path.join(abstracts_dir, file)
            try:
                with gzip.open(file_path, "rt", encoding="utf-8") as f:
                    for line in f:
                        try:
                            record = json.loads(line.strip())
                            abstract = record.get("abstract")
                            if abstract and isinstance(abstract, str):
                                abstract = abstract.strip()
                                if abstract and is_strictly_english(abstract, min_length=20, min_words=3):
                                    buffer.append(abstract + "\n")
                                    total_written += 1
                        except json.JSONDecodeError:
                            continue

                        if len(buffer) >= buffer_size:
                            out_f.writelines(buffer)
                            buffer.clear()
            except Exception as e:
                print(f" Error reading {file}: {e}")
                continue

        if buffer:
            out_f.writelines(buffer)

    print(f" Step 2 complete: {total_written:,} abstracts written to {abstracts_output_file}")

# --- Step 3: Merge Titles and Abstracts ---
def merge_titles_and_abstracts(titles_file_path, abstracts_file_path, output_file_path):
    """
    Merge titles and abstracts from separate files into a single output file.
    
    This function reads titles and abstracts line by line from their respective files
    and writes them as pairs to a combined output file. Each title-abstract pair is
    written with the title on one line followed by its abstract on the next line.
    
    Args:
        titles_file_path (str): Path to the file containing filtered titles
        abstracts_file_path (str): Path to the file containing filtered abstracts
        output_file_path (str): Path where the merged output will be written
        
    The function:
    1. Opens all three files (titles, abstracts, and output) simultaneously
    2. Reads titles and abstracts line by line
    3. Cleans each title and abstract by removing newlines and extra whitespace
    4. Writes each title-abstract pair to the output file
    5. Keeps track of the number of pairs processed
    
    Returns:
        None. Results are written directly to output_file_path.
        Progress and completion messages are printed to stdout.
        
    Note:
        - Assumes titles and abstracts files have matching entries line by line
        - Stops processing when either file runs out of lines
        - Uses buffered writing for performance optimization
    """
    print("\n Merging titles and abstracts into final output...")
    with open(titles_file_path, "r", encoding="utf-8") as title_f, \
         open(abstracts_file_path, "r", encoding="utf-8") as abstract_f, \
         open(output_file_path, "w", encoding="utf-8", buffering=16384) as output_f:

        count = 0
        while True:
            title = title_f.readline()
            abstract = abstract_f.readline()

            if not title or not abstract:
                break

            title_clean = title.replace('\n', ' ').strip()
            abstract_clean = abstract.replace('\n', ' ').strip()

            output_f.write(f"{title_clean}\r\n")
            output_f.write(f"{abstract_clean}\r\n")
            count += 1

    print(f" Step 3 complete: {count:,} title-abstract pairs written to {output_file_path}")

# --- Main ---
if __name__ == "__main__":
    print("Starting 3-step title + abstract processing pipeline...\n")
    process_titles()
    process_abstracts()
    merge_titles_and_abstracts(titles_output_file, abstracts_output_file, merged_output_file)
    print("\n Completed!")


Starting 3-step title + abstract processing pipeline...

 Processing English titles...


KeyboardInterrupt: 

In [4]:
def print_first_100_lines(filepath):
    """
    Print the first 100 lines from a text file as-is.

    Args:
        filepath (str): Full path to the merged dataset file
    """
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for i in range(100):
                line = f.readline()
                if not line:
                    print(f"\n[Reached end of file after {i} lines]\n")
                    break
                print(line.strip())
    except FileNotFoundError:
        print(f"⚠️ File not found: {filepath}")
    except Exception as e:
        print(f"⚠️ Error reading file: {e}")


In [ ]:
merged_output_file = r"C://Users//Faisal Ramzan//Desktop//kmi_project_cso//paper_dataset//paper_dataset.txt"
print_first_100_lines(merged_output_file)


## Experiments

In [7]:
from gensim.models import KeyedVectors

def load_word2vec_model(path: str) -> KeyedVectors:
    """Load the full Word2Vec binary model (no limit)."""
    print(f"Loading Word2Vec model from: {path} ...")
    model = KeyedVectors.load_word2vec_format(path, binary=True)
    model.fill_norms()  # precompute norms for similarity queries
    print(f"Loaded model successfully.")
    return model

def get_vocab_size(model: KeyedVectors) -> int:
    """Return the number of tokens in the model vocabulary."""
    return len(model.key_to_index)

if __name__ == "__main__":
    model_path = "./w2v_model/264M_cp_32.bin"

    # Load full model (⚠️ requires more RAM, may take several minutes)
    model = load_word2vec_model(model_path)

    vocab_size = get_vocab_size(model)
    print(f"\nExact vocabulary size: {vocab_size:,}")

    # Example: show first 20 tokens
    top_tokens = list(model.key_to_index.keys())[:20]
    print("First 20 tokens in vocab:")
    print(", ".join(top_tokens))


Loading Word2Vec model from: ./w2v_model/264M_cp_32.bin ...
Loaded model successfully.

Exact vocabulary size: 2,074,429
First 20 tokens in vocab:
the, of, and, in, to, a, for, is, with, on, that, by, was, as, this, are, were, from, an, at


In [31]:
from typing import List
from gensim.models import KeyedVectors

# ---------------------------
# Helpers: load + vocabulary
# ---------------------------
def load_word2vec_model(path: str, limit: int | None = None) -> KeyedVectors:
    """
    Load a Word2Vec binary model. If 'limit' is set, loads only the top-N
    most frequent vectors (much faster and lighter on RAM).
    """
    print(f"Loading Word2Vec model from: {path} (limit={limit}) ...")
    model = KeyedVectors.load_word2vec_format(path, binary=True, limit=limit)
    # Precompute L2 norms for fast cosine similarities
    model.fill_norms()
    print(f"Loaded. Vocab size: {len(model.key_to_index):,}")
    return model

def get_vocab_size(model: KeyedVectors) -> int:
    """Return the number of tokens in the model's vocabulary."""
    return len(model.key_to_index)

def get_top_tokens(model: KeyedVectors, n: int = 20) -> List[str]:
    """
    Return the first 'n' tokens by their index order.
    In word2vec, lower index usually means higher frequency.
    """
    # key_to_index is {token: idx}; sort by idx and take top n
    return [tok for tok, _ in sorted(model.key_to_index.items(), key=lambda kv: kv[1])[:n]]

# ---------------------------
# Main usage example
# ---------------------------
if __name__ == "__main__":
    model_path = "./w2v_model/264M_cp_32.bin"

    # Adjust this depending on your RAM/speed needs.
    # - None  -> load full vocab (can be VERY large)
    # - 500_000 -> load top 500k tokens
    w2v_limit = 2500_000
    # w2v_limit = 2074429
    model = load_word2vec_model(model_path, limit=w2v_limit)

    # Vocabulary info
    vocab_size = get_vocab_size(model)
    print(f"\nVocabulary size: {vocab_size:,}")

    top_tokens = get_top_tokens(model, n=20)
    print("Top 20 tokens (by index/frequency):")
    print(", ".join(top_tokens))

    # Similarity test
    test_word = "data_mining"  # try: "image_segmentation", "web", "learning", "science", "semantic_web"
    if test_word in model.key_to_index:
        similar_words = model.most_similar(test_word, topn=20)
        print(f"\nTest word: {test_word}")
        for word, score in similar_words:
            print(f"{word}: {score:.4f}")
    else:
        print(f"\n'{test_word}' not in vocabulary")


Loading Word2Vec model from: ./w2v_model/264M_cp_32.bin (limit=2500000) ...
Loaded. Vocab size: 2,074,429

Vocabulary size: 2,074,429
Top 20 tokens (by index/frequency):
the, of, and, in, to, a, for, is, with, on, that, by, was, as, this, are, were, from, an, at

Test word: data_mining
knowledge_discovery: 0.8005
data_mining_techniques: 0.7967
association_rule_mining: 0.7586
text_mining: 0.7506
big_data: 0.7434
data_mining_algorithms: 0.7434
data_mining_dm: 0.7397
datamining: 0.7386
association_rules: 0.7386
web_mining: 0.7236
knowledge_discovery_in_databases_kdd: 0.7218
web_usage_mining: 0.7208
data_mining_methods: 0.7200
data_mining_process: 0.7164
knowledge_discovery_in_databases: 0.7155
data_mining_algorithm: 0.7107
association_rules_mining: 0.7105
machine_learning: 0.7084
data_mining_applications: 0.6924
web_data_mining: 0.6907


In [ ]:
# download the previous version of the token-to-cso-combined.json file
import os
import urllib.request

url = "https://cso.kmi.open.ac.uk/download/token-to-cso-combined.json"
dest_path = "./token-to-cso-combined.json"

# Make sure folder exists
os.makedirs(os.path.dirname(dest_path) or ".", exist_ok=True)

print(f"[*] Downloading from {url}")
urllib.request.urlretrieve(url, dest_path)
print(f"[*] File saved to {dest_path}")


[*] Downloading from https://cso.kmi.open.ac.uk/download/token-to-cso-combined.json
[*] File saved to ./token-to-cso-combined.json


In [17]:
import sys
from rapidfuzz.distance import Levenshtein
sys.path.append(r"C:\Users\Faisal Ramzan\Desktop\kmi_project_cso\cso-reader-main\cso_reader")
from ontology import Ontology

onto = Ontology()
query = "semantic web"
results = sorted([(t, Levenshtein.normalized_similarity(t.lower(), query.lower())) 
                  for t in onto.topics], key=lambda x: x[1], reverse=True)[:20]

for topic, score in results:
    print(f"{topic}  ({score:.3f})")


Computer Science Ontology loaded.
semantic web  (1.000)
semantic wiki  (0.769)
semantic gap  (0.750)
semantic wikis  (0.714)
semantic data  (0.692)
semantics  (0.667)
semantic search  (0.667)
semantic memory  (0.667)
semantic images  (0.667)
semantic levels  (0.667)
semantic concept  (0.625)
semantic service  (0.625)
semantic desktop  (0.625)
semantic web service  (0.600)
semantic features  (0.588)
semantic metadata  (0.588)
semantic security  (0.588)
semantic dementia  (0.588)
semantic web services  (0.571)
semantic priming  (0.562)
